<a href="https://colab.research.google.com/github/nbrookes80-spec/LS_Based_Engine_Repository/blob/master/avatarify.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Avatarify Colab Server

This Colab notebook is for running Avatarify rendering server. It allows you to run Avatarify on your computer **without GPU** in this way:

1. When this notebook is executed, it starts listening for incoming requests from your computer;
1. You start the client on your computer and it connects to the notebook and starts sending requests;
1. This notebooks receives the requests from your computer, renders avatar images and sends them back;

To this end, all the heavy work is offloaded from your computer to this notebook so you don't need to have a beafy hardware on your PC anymore.


## Start the server
Run the cells below (Shift+Enter) sequentially and pay attention to the hints and instructions included in this notebook.

At the end you will get a command for running the client on your computer.

## Start the client

Make sure you have installed the latest version of Avatarify on your computer. Refer to the [README](https://github.com/alievk/avatarify#install) for the instructions.

When it's ready execute this notebook and get the command for running the client on your computer.


### Technical details

The client on your computer connects to the server via `ngrok` TCP tunnel or a reverse `ssh` tunnel.

`ngrok`, while easy to use, can induce a considerable network lag ranging from dozens of milliseconds to a second. This can lead to a poor experience.

A more stable connection could be established using a reverse `ssh` tunnel to a host with a public IP, like an AWS `t3.micro` (free) instance. This notebook provides a script for creating a tunnel, but launching an instance in a cloud is on your own (find the manual below).

# Install

### Avatarify
Follow the steps below to clone Avatarify and install the dependencies.

In [123]:
!cd /content
!rm -rf *

In [124]:
!git clone https://github.com/alievk/avatarify.git

Cloning into 'avatarify'...
remote: Enumerating objects: 1514, done.
remote: Total 1514 (delta 0), reused 0 (delta 0), pack-reused 1514 (from 1)
Receiving objects: 100% (1514/1514), 5.69 MiB | 17.75 MiB/s, done.
Resolving deltas: 100% (967/967), done.


In [125]:
cd avatarify

/content/avatarify/avatarify


In [126]:
!git clone https://github.com/alievk/first-order-model.git fomm
!pip install face-alignment msgpack_numpy pyyaml

Cloning into 'fomm'...
remote: Enumerating objects: 211, done.
remote: Total 211 (delta 0), reused 0 (delta 0), pack-reused 211 (from 1)
Receiving objects: 100% (211/211), 58.16 MiB | 24.42 MiB/s, done.
Resolving deltas: 100% (108/108), done.


In [127]:
!scripts/download_data.sh

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  228M  100  228M    0     0   154M      0  0:00:01  0:00:01 --:--:--  154M
Expected checksum: 8a45a24037871c045fbb8a6a8aa95ebc
Found checksum:    8a45a24037871c045fbb8a6a8aa95ebc  vox-adv-cpk.pth.tar


### ngrok
Follow the steps below to setup ngrok. You will also need to sign up on the ngrok site and get your authtoken (free).


In [128]:
# Download ngrok
!scripts/get_ngrok.sh

ngrok is not found, installing...
Done!


# Run
Start here if the runtime was restarted after installation.

In [129]:
%cd /content/avatarify/avatarify
!ls

/content/avatarify/avatarify
afy		 LICENSE.md			  run.sh
avatarify.ipynb  ngrok				  run_windows.bat
avatars		 ngrok-v3-stable-linux-amd64.tgz  scripts
config.yaml	 README.md			  var
Dockerfile	 requirements_client.txt	  vox-adv-cpk.pth.tar
docs		 requirements.txt
fomm		 run_mac.sh


In [130]:
#!git pull origin

In [131]:
from subprocess import Popen, PIPE
import shlex
import json
import time


def run_with_pipe(command):
  commands = list(map(shlex.split,command.split("|")))
  ps = Popen(commands[0], stdout=PIPE, stderr=PIPE)
  for command in commands[1:]:
    ps = Popen(command, stdin=ps.stdout, stdout=PIPE, stderr=PIPE)
  return ps.stdout.readlines()


def get_tunnel_adresses():
  info = run_with_pipe("curl http://localhost:4040/api/tunnels")
  assert info

  info = json.loads(info[0])
  for tunnel in info['tunnels']:
    url = tunnel['public_url']
    port = url.split(':')[-1]
    local_port = tunnel['config']['addr'].split(':')[-1]
    print(f'{url} -> {local_port} [{tunnel["name"]}]')
    if tunnel['name'] == 'input':
      in_addr = url
    elif tunnel['name'] == 'output':
      out_addr = url
    else:
      print(f'unknown tunnel: {tunnel["name"]}')

  return in_addr, out_addr

In [132]:
# Input and output ports for communication
local_in_port = 5557
local_out_port = 5558

# Start the worker


In [133]:
# (Re)Start the worker
with open('/tmp/run.txt', 'w') as f:
  ps = Popen(
      shlex.split(f'./run.sh --is-worker --in-port {local_in_port} --out-port {local_out_port} --no-vcam --no-conda'),
      stdout=f, stderr=f)
  time.sleep(3)

This command should print lines if the worker is successfully started

In [134]:
!ps aux | grep 'python3 afy/cam_fomm.py' | grep -v grep | tee /tmp/ps_run
!if [[ $(cat /tmp/ps_run | wc -l) == "0" ]]; then echo "Worker failed to start"; cat /tmp/run.txt; else echo "Worker started"; fi

root       31946  105  2.1 1121656 282792 ?      Rl   11:11   0:03 python3 afy/cam_fomm.py --config fomm/config/vox-adv-256.yaml --checkpoint vox-adv-cpk.pth.tar --virt-cam 9 --relative --adapt_scale --is-worker --in-port 5557 --out-port 5558 --no-stream
Worker started


# Open ngrok tunnel

#### Get ngrok token
Go to https://dashboard.ngrok.com/auth/your-authtoken (sign up if required), copy your authtoken and put it below.

In [135]:
# Paste your authtoken here in quotes
authtoken = "ng-3KXn63Y6GlNb6HJJCkXBnpNwMZt-g1-jCyXZxFNunNQ22EhGQhHw7giKGfdrZ6E"

Set your region

Code | Region
--- | ---
us | United States
eu | Europe
ap | Asia/Pacific
au | Australia
sa | South America
jp | Japan
in | India

In [136]:
# Set your region here in quotes
region = "au"

In [137]:
config =\
f"""
version: 2
authtoken: {authtoken}
region: {region}
console_ui: False
tunnels:
  input:
    addr: {local_in_port}
    proto: tcp
  output:
    addr: {local_out_port}
    proto: tcp
"""

with open('ngrok.conf', 'w') as f:
  f.write(config)

In [138]:
!pkill -f cam_fomm; pkill -f ngrok; sleep 1
%cd /content
!rm -rf /content/avatarify
!git clone https://github.com/alievk/avatarify.git
%cd /content/avatarify
!git clone https://github.com/alievk/first-order-model.git fomm
!scripts/download_data.sh
!scripts/get_ngrok.sh
!ls /content/avatarify /content/avatarify/scripts

^C
/content
Cloning into 'avatarify'...
remote: Enumerating objects: 1514, done.
remote: Total 1514 (delta 0), reused 0 (delta 0), pack-reused 1514 (from 1)
Receiving objects: 100% (1514/1514), 5.69 MiB | 18.14 MiB/s, done.
Resolving deltas: 100% (967/967), done.
/content/avatarify
Cloning into 'fomm'...
remote: Enumerating objects: 211, done.
remote: Total 211 (delta 0), reused 0 (delta 0), pack-reused 211 (from 1)
Receiving objects: 100% (211/211), 58.16 MiB | 25.83 MiB/s, done.
Resolving deltas: 100% (108/108), done.
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  228M  100  228M    0     0   234M      0 --:--:-- --:--:-- --:--:--  234M
Expected checksum: 8a45a24037871c045fbb8a6a8aa95ebc
Found checksum:    8a45a24037871c045fbb8a6a8aa95ebc  vox-adv-cpk.pth.tar
ngrok is not found, installing...
Done!
/content/avatarify:
afy		 LICENSE.md			  run.sh
avatarify.ipynb  ngrok		

In [139]:
%cd /content/avatarify
# (Re)Open tunnel
ps = Popen('./scripts/open_tunnel_ngrok.sh', stdout=PIPE, stderr=PIPE)
time.sleep(3)

/content/avatarify


In [140]:
# Get tunnel addresses
try:
  in_addr, out_addr = get_tunnel_adresses()
  print("Tunnel opened")
except Exception as e:
  [print(l.decode(), end='') for l in ps.stdout.readlines()]
  print("Something went wrong, reopen the tunnel")

Opening tunnel
Something went wrong, reopen the tunnel


In [141]:
!cat /content/avatarify/scripts/open_tunnel_ngrok.sh; cd /content/avatarify && timeout 8 ./ngrok start --all --config ngrok.conf --log=stdout 2>&1 | head -25

#!/usr/bin/env bash

cmd="./ngrok start --all --config ngrok.conf"

kill -9 $(ps aux | grep $cmd | awk '{print $2}') 2> /dev/null

echo Opening tunnel
$cmd

t=2026-10-11T11:11:58+0000 lvl=info msg="open config file" path=/content/avatarify/ngrok.conf err="open /content/avatarify/ngrok.conf: no such file or directory"
ERROR:  Error reading configuration file 'ngrok.conf': open /content/avatarify/ngrok.conf: no such file or directory


### [Optional] AWS proxy
Alternatively you can create a ssh reverse tunnel to an AWS `t3.micro` instance (it's free). It has lower latency than ngrok.

1. In your AWS console go to Services -> EC2 -> Instances -> Launch Instance;
1. Choose `Ubuntu Server 18.04 LTS` AMI;
1. Choose `t3.micro` instance type and press Review and launch;
1. Confirm your key pair and press Launch instances;
1. Go to the security group of this instance and edit inbound rules. Add TCP ports 5557 and 5558 and set Source to Anywhere. Press Save rules;
1. ssh into the instance (you can find the command in the Instances if you click on the Connect button) and add this line in the end of `/etc/ssh/sshd_config`:
```
GatewayPorts yes
```
then restart `sshd`
```
sudo service sshd restart
```
1. Copy your `key_pair.pem` by dragging and dropping it into avatarify folder in this notebook;
1. Use the command below to open the tunnel;
1. Start client with a command (substitute `run_mac.sh` with `run_windows.bat` or `run.sh`)
```
./run_mac.sh --is-client --in-addr tcp://instace.compute.amazonaws.com:5557 --out-addr tcp://instance.compute.amazonaws.com:5558
```

In [142]:
# Open reverse ssh tunnel (uncomment line below)
# !./scripts/open_tunnel_ssh.sh key_pair.pem ubuntu@instance.compute.amazonaws.com

# Start the client
When you run the cell below it will print a command. Run this command on your computer:

1. Open a terminal (in Windows open `Anaconda Prompt`);
2. Change working directory to the `avatarify` directory:</br>
* Windows (change `C:\path\to\avatarify` to your path)</br>
`cd C:\path\to\avatarify`</br></br>
* Mac/Linux (change `/path/to/avatarify` to your path)</br>
`cd /path/to/avatarify`
3. Copy-paste to the terminal the command below and run;
4. It can take some time to connect (usually up to 10 seconds). If the preview window doesn't appear in a minute or two, look for the errors above in this notebook and report in the [issues](https://github.com/alievk/avatarify/issues) or [Slack](https://join.slack.com/t/avatarify/shared_invite/zt-dyoqy8tc-~4U2ObQ6WoxuwSaWKKVOgg).

In [152]:
print('Copy-paste to the terminal the command below and run (press Enter)\n')
print('Mac:')
print(f'./run_mac.sh --is-client --in-addr {in_addr} --out-addr {out_addr}')
print('\nWindows:')
print(f'run_windows.bat --is-client --in-addr {in_addr} --out-addr {out_addr}')
print('\nLinux:')
print(f'./run.sh --is-client --in-addr {in_addr} --out-addr {out_addr}')

Copy-paste to the terminal the command below and run (press Enter)

Mac:


NameError: name 'in_addr' is not defined

In [149]:
!rm -f /content/avatarify/proxy_key /content/avatarify/proxy_key.pub; ssh-keygen -t ed25519 -N "" -C avatarify-colab -f /content/avatarify/proxy_key -q; cat /content/avatarify/proxy_key.pub

ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIFlmGHkMUp3R65YUP6z+T/wyGeukaQfAlCj+gtNUmCXr avatarify-colab


In [150]:
!cat /content/avatarify/scripts/open_tunnel_ssh.sh; ls -la /content/avatarify/proxy_key*

#!/usr/bin/env bash

remote_port_in=5557
remote_port_out=5558
port_in=5557
port_out=5558

KEY=$1
REMOTE_HOST=$2

if [[ -z $KEY ]] || [[ -z $REMOTE_HOST ]]; then
	echo Usage: open_tunnel_ssh.sh path_to_key.pem remote_host
	exit
fi

port_out=5558

chmod 400 $KEY

cmd_in="ssh -f -N -T -R $remote_port_in:localhost:$port_in -i $KEY -o StrictHostKeyChecking=no $REMOTE_HOST"
cmd_out="ssh -f -N -T -R $remote_port_out:localhost:$port_out -i $KEY -o StrictHostKeyChecking=no $REMOTE_HOST"

kill -9 $(ps aux | grep "$cmd_in" | awk '{print $2}') 2> /dev/null
kill -9 $(ps aux | grep "$cmd_out" | awk '{print $2}') 2> /dev/null

echo Open tunnels
set +x
$cmd_in
$cmd_out

-rw------- 1 root root 411 Oct 11 11:15 /content/avatarify/proxy_key
-rw-r--r-- 1 root root  97 Oct 11 11:15 /content/avatarify/proxy_key.pub


In [151]:
%cd /content/avatarify
!./scripts/open_tunnel_ssh.sh /content/avatarify/proxy_key ubuntu@13.210.132.155
!sleep 3; ps aux | grep "ssh -f" | grep -v grep; pgrep -af cam_fomm | head -3

/content/avatarify
Open tunnels
ubuntu@13.210.132.155: Permission denied (publickey).
ubuntu@13.210.132.155: Permission denied (publickey).
33067 /bin/bash -c sleep 3; ps aux | grep "ssh -f" | grep -v grep; pgrep -af cam_fomm | head -3


# Logs

If something doesn't work as expected, please run the cells below and include the logs in your report.

In [147]:
#@title
!cat ./var/log/cam_fomm.log | head -100

cat: ./var/log/cam_fomm.log: No such file or directory


In [146]:
#@title
!cat ./var/log/recv_worker.log | tail -100

cat: ./var/log/recv_worker.log: No such file or directory


In [145]:
#@title
!cat ./var/log/predictor_worker.log | tail -100

cat: ./var/log/predictor_worker.log: No such file or directory


In [144]:
#@title
!cat ./var/log/send_worker.log | tail -100

cat: ./var/log/send_worker.log: No such file or directory
